In [ ]:
"""
Bachelor / Bachelorette — Descriptive Statistical Analysis
==========================================================
Four figures:
  Fig 1 — Summary statistics table + binary feature prevalence bar chart
  Fig 2 — Contestant survival curves by show (Bachelor vs Bachelorette)
  Fig 3 — Elimination rate by week + date activity overlay
  Fig 4 — Avg week of elimination by contestant feature profile (box plots)
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy import stats

# ── palette ──────────────────────────────────────────────────────────────────
ROSE     = "#C9184A"
TEAL     = "#2A9D8F"
GOLD     = "#F4A261"
LILAC    = "#9B5DE5"
CHARCOAL = "#1D1B1E"
SOFT_RED = "#FF4D6D"
BG       = "#FFF8F9"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    BG,
    "axes.edgecolor":    "#DDCCCC",
    "axes.labelcolor":   CHARCOAL,
    "xtick.color":       CHARCOAL,
    "ytick.color":       CHARCOAL,
    "text.color":        CHARCOAL,
    "font.family":       "DejaVu Sans",
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.color":        "#DDCCCC",
})


# ════════════════════════════════════════════════════════════════════════════
# DATA PREP
# ════════════════════════════════════════════════════════════════════════════

def load(path: str):
    df = pd.read_csv(path)

    # ── Contestant-level features (take max/any across their 10 rows) ──
    feats = (
        df.groupby(["SHOW", "SEASON", "CONTESTANT"])
        .agg(
            ever_date_rose      = ("DATE_ROSE",            "max"),
            ever_one_on_one     = ("ONE_ON_ONE",           "max"),
            ever_weekly_rose    = ("WEEKLY_ROSE",          "max"),
            ever_has_date       = ("HAS_DATE",             "max"),
            first_impression    = ("FIRST_IMPRESSION_ROSE","max"),
        )
        .reset_index()
    )

    # ── Week of elimination (or 11 = reached finale) ──
    elim = (
        df[df["ELIMINATED_THIS_WEEK"]]
        .groupby(["SHOW", "SEASON", "CONTESTANT"])["WEEK"]
        .min()
        .reset_index()
        .rename(columns={"WEEK": "elim_week"})
    )
    cx = feats.merge(elim, on=["SHOW", "SEASON", "CONTESTANT"], how="left")
    cx["elim_week"] = cx["elim_week"].fillna(11)   # 11 = finalist / never eliminated
    cx["finalist"]  = cx["elim_week"] == 11

    # ── Weekly aggregates ──
    weekly = (
        df.groupby("WEEK")
        .agg(
            n_alive      = ("ALIVE",               "sum"),
            n_elim       = ("ELIMINATED_THIS_WEEK","sum"),
            n_has_date   = ("HAS_DATE",            "sum"),
            n_date_rose  = ("DATE_ROSE",           "sum"),
            n_one_on_one = ("ONE_ON_ONE",          "sum"),
        )
        .reset_index()
    )
    weekly["elim_rate"]      = weekly["n_elim"] / (weekly["n_alive"] + weekly["n_elim"])
    weekly["date_rose_rate"] = weekly["n_date_rose"] / weekly["n_has_date"].replace(0, np.nan)

    # ── Survival curves: % alive each week, by show ──
    alive = df.groupby(["SHOW", "SEASON", "WEEK"])["ALIVE"].sum().reset_index()
    w1    = (
        alive[alive["WEEK"] == 1][["SHOW", "SEASON", "ALIVE"]]
        .rename(columns={"ALIVE": "start"})
    )
    alive = alive.merge(w1, on=["SHOW", "SEASON"])
    alive["pct_alive"] = alive["ALIVE"] / alive["start"]

    return df, cx, weekly, alive


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Summary stats table + feature prevalence
# ════════════════════════════════════════════════════════════════════════════

def fig1_summary(df, cx, out):
    fig = plt.figure(figsize=(15, 6.5))
    fig.patch.set_facecolor(BG)
    gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.38)
    ax_tbl = fig.add_subplot(gs[0])
    ax_bar = fig.add_subplot(gs[1])

    fig.suptitle(
        "Bachelor / Bachelorette — Descriptive Summary Statistics",
        fontsize=14, fontweight="bold", color=CHARCOAL
    )

    # ── left: summary table ──
    ax_tbl.axis("off")

    n_shows    = df["SHOW"].nunique()
    n_seasons  = df.groupby(["SHOW","SEASON"]).ngroups
    n_cx       = cx.shape[0]                          # unique contestants
    avg_cast   = cx.groupby(["SHOW","SEASON"]).size().mean()
    pct_elim   = (~cx["finalist"]).mean() * 100
    avg_elim_wk= cx[~cx["finalist"]]["elim_week"].mean()
    pct_date   = cx["ever_has_date"].mean() * 100
    pct_1on1   = cx["ever_one_on_one"].mean() * 100
    pct_drose  = cx["ever_date_rose"].mean() * 100
    pct_wrose  = cx["ever_weekly_rose"].mean() * 100
    pct_fi     = cx["first_impression"].mean() * 100

    rows = [
        ("Shows",                   f"{n_shows}"),
        ("Seasons",                 f"{n_seasons}"),
        ("Unique contestants",      f"{n_cx:,}"),
        ("Avg cast size / season",  f"{avg_cast:.1f}"),
        ("", ""),
        ("Ever eliminated",         f"{pct_elim:.1f}%"),
        ("Reached finale",          f"{100-pct_elim:.1f}%"),
        ("Avg elim week (excl. finalists)", f"{avg_elim_wk:.2f}"),
        ("", ""),
        ("Ever had a date",         f"{pct_date:.1f}%"),
        ("Ever had 1-on-1 date",    f"{pct_1on1:.1f}%"),
        ("Ever received date rose", f"{pct_drose:.1f}%"),
        ("Ever received weekly rose",f"{pct_wrose:.1f}%"),
        ("Received 1st-impression rose", f"{pct_fi:.1f}%"),
    ]

    for i, (label, val) in enumerate(rows):
        y = 1 - i / len(rows)
        if label == "":
            continue
        bg_c = "#FFF0F3" if i % 2 == 0 else BG
        ax_tbl.axhspan(y - 1/len(rows), y, xmin=0, xmax=1, color=bg_c, zorder=0)
        ax_tbl.text(0.02, y - 0.5/len(rows), label,
                    va="center", fontsize=10, color=CHARCOAL)
        ax_tbl.text(0.97, y - 0.5/len(rows), val,
                    va="center", ha="right", fontsize=10,
                    fontweight="bold", color=ROSE)

    ax_tbl.set_title("Dataset Overview", fontsize=11, color=CHARCOAL, pad=8)
    ax_tbl.set_xlim(0, 1); ax_tbl.set_ylim(0, 1)

    # ── right: feature prevalence by finalist status ──
    features  = ["ever_has_date", "ever_one_on_one", "ever_date_rose",
                 "ever_weekly_rose", "first_impression"]
    labels_f  = ["Had a Date", "1-on-1 Date", "Date Rose",
                 "Weekly Rose", "1st Imp. Rose"]

    fin   = cx[cx["finalist"]]
    elim  = cx[~cx["finalist"]]

    fin_rates  = [fin[f].mean()  * 100 for f in features]
    elim_rates = [elim[f].mean() * 100 for f in features]

    x   = np.arange(len(features))
    w   = 0.35
    b1  = ax_bar.bar(x - w/2, elim_rates, w, color=ROSE,  label="Eliminated",  alpha=0.88)
    b2  = ax_bar.bar(x + w/2, fin_rates,  w, color=TEAL,  label="Finalist",    alpha=0.88)

    for bar in list(b1) + list(b2):
        ax_bar.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 1,
                    f"{bar.get_height():.0f}%",
                    ha="center", va="bottom", fontsize=8, color=CHARCOAL)

    ax_bar.set_xticks(x)
    ax_bar.set_xticklabels(labels_f, fontsize=9)
    ax_bar.set_ylabel("% of contestants", fontsize=10)
    ax_bar.set_title("Feature Prevalence: Eliminated vs. Finalists", fontsize=11, color=CHARCOAL)
    ax_bar.legend(fontsize=9)
    ax_bar.set_ylim(0, 115)

    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"  ✓ {out}")


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Survival curves by show
# ════════════════════════════════════════════════════════════════════════════

def fig2_survival(alive, cx, out):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
    fig.patch.set_facecolor(BG)
    fig.suptitle(
        "Contestant Survival Curves — % of Opening Cast Still Alive Each Week",
        fontsize=13, fontweight="bold", color=CHARCOAL
    )

    show_color = {"Bachelor": ROSE, "Bachelorette": TEAL}

    for ax, show in zip(axes, ["Bachelor", "Bachelorette"]):
        sub = alive[alive["SHOW"] == show]
        seasons = sub["SEASON"].unique()

        # individual season curves (light)
        for s in seasons:
            sg = sub[sub["SEASON"] == s].sort_values("WEEK")
            ax.plot(sg["WEEK"], sg["pct_alive"] * 100,
                    color=show_color[show], alpha=0.18, lw=1.2)

        # mean curve
        mean_curve = sub.groupby("WEEK")["pct_alive"].mean() * 100
        ax.plot(mean_curve.index, mean_curve.values,
                color=show_color[show], lw=2.8, label="Season mean",
                zorder=5)

        # 25/75 percentile band
        lo = sub.groupby("WEEK")["pct_alive"].quantile(0.25) * 100
        hi = sub.groupby("WEEK")["pct_alive"].quantile(0.75) * 100
        ax.fill_between(mean_curve.index, lo, hi,
                        color=show_color[show], alpha=0.15,
                        label="IQR across seasons")

        # annotate median survivor count at each week
        for wk, pct in mean_curve.items():
            ax.text(wk, pct + 2.5, f"{pct:.0f}%",
                    ha="center", fontsize=7, color=show_color[show])

        ax.set_title(f"The {show}", fontsize=12, color=show_color[show])
        ax.set_xlabel("Week", fontsize=10)
        ax.set_ylabel("% of cast still alive", fontsize=10)
        ax.set_xticks(range(1, 11))
        ax.set_ylim(0, 115)
        ax.legend(fontsize=9)

        # summary stats box
        finalist_pct = cx[cx["SHOW"] == show]["finalist"].mean() * 100
        avg_wk = cx[(cx["SHOW"] == show) & ~cx["finalist"]]["elim_week"].mean()
        ax.text(0.97, 0.97,
                f"Seasons: {len(seasons)}\nFinalist rate: {finalist_pct:.1f}%\nAvg elim week: {avg_wk:.2f}",
                transform=ax.transAxes, va="top", ha="right",
                fontsize=8.5, color=CHARCOAL,
                bbox=dict(boxstyle="round,pad=0.4", fc=BG, ec="#DDCCCC", alpha=0.9))

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"  ✓ {out}")


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Weekly elimination rate + date activity overlay
# ════════════════════════════════════════════════════════════════════════════

def fig3_weekly_dynamics(weekly, out):
    fig, ax1 = plt.subplots(figsize=(12, 5.5))
    fig.patch.set_facecolor(BG)

    weeks = weekly["WEEK"]

    # ── bar: elimination rate ──
    bars = ax1.bar(weeks, weekly["elim_rate"] * 100,
                   color=ROSE, alpha=0.75, label="Elimination rate (%)",
                   zorder=2)
    for bar, val in zip(bars, weekly["elim_rate"]):
        ax1.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f"{val*100:.1f}%",
                 ha="center", va="bottom", fontsize=8.5, color=CHARCOAL)

    ax1.set_xlabel("Week", fontsize=11)
    ax1.set_ylabel("Elimination rate (%)", fontsize=11, color=ROSE)
    ax1.tick_params(axis="y", labelcolor=ROSE)
    ax1.set_xticks(weeks)
    ax1.set_ylim(0, 50)

    # ── line: date-rose rate among those on dates (secondary axis) ──
    ax2 = ax1.twinx()
    ax2.set_facecolor(BG)
    ax2.plot(weeks, weekly["date_rose_rate"] * 100,
             color=TEAL, lw=2.5, marker="o", markersize=7,
             label="Date-rose rate (% of dates)", zorder=5)
    ax2.set_ylabel("Date-rose rate (% of daters)", fontsize=11, color=TEAL)
    ax2.tick_params(axis="y", labelcolor=TEAL)
    ax2.set_ylim(0, 100)

    # ── annotate n_has_date ──
    for _, row in weekly.iterrows():
        ax1.text(row["WEEK"], -3.5, f"n={int(row['n_has_date'])}",
                 ha="center", fontsize=7.5, color=CHARCOAL, style="italic")
    ax1.text(0.5, -0.09, "n = total dates that week (pooled across all seasons)",
             transform=ax1.transAxes, ha="center", fontsize=8, color="#888")

    # ── combined legend ──
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=9, loc="upper right")

    fig.suptitle(
        "Weekly Elimination Rate & Date-Rose Rate — Pooled Across All Seasons",
        fontsize=13, fontweight="bold", color=CHARCOAL
    )
    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"  ✓ {out}")


# ════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Elimination week by feature profile (box plots + t-test)
# ════════════════════════════════════════════════════════════════════════════

def fig4_feature_boxplots(cx, out):
    # Only eliminated contestants (elim_week < 11)
    elim = cx[cx["elim_week"] < 11].copy()

    features = [
        ("ever_date_rose",   "Received\nDate Rose",   TEAL, ROSE),
        ("ever_one_on_one",  "Had 1-on-1\nDate",      TEAL, ROSE),
        ("first_impression", "Received 1st\nImp. Rose",TEAL, ROSE),
        ("ever_has_date",    "Had Any\nDate",          TEAL, ROSE),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(15, 6), sharey=True)
    fig.patch.set_facecolor(BG)
    fig.suptitle(
        "Week of Elimination by Contestant Feature — Eliminated Contestants Only\n"
        "(Finalists excluded; week 11 = finale)",
        fontsize=13, fontweight="bold", color=CHARCOAL
    )

    for ax, (col, title, c_yes, c_no) in zip(axes, features):
        grp_no  = elim[~elim[col]]["elim_week"].dropna()
        grp_yes = elim[elim[col]]["elim_week"].dropna()

        bp = ax.boxplot(
            [grp_no, grp_yes],
            patch_artist=True,
            widths=0.5,
            medianprops=dict(color="white", lw=2.5),
            whiskerprops=dict(color=CHARCOAL),
            capprops=dict(color=CHARCOAL),
            flierprops=dict(marker="o", color=CHARCOAL, alpha=0.4, markersize=4),
        )
        bp["boxes"][0].set_facecolor(c_no)
        bp["boxes"][1].set_facecolor(c_yes)
        bp["boxes"][0].set_alpha(0.75)
        bp["boxes"][1].set_alpha(0.75)

        # t-test
        if len(grp_no) > 1 and len(grp_yes) > 1:
            t, p = stats.ttest_ind(grp_no, grp_yes, equal_var=False)
            sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
            ax.text(1.5, 10.6, f"p={p:.3f} {sig}",
                    ha="center", fontsize=9, color=CHARCOAL,
                    bbox=dict(boxstyle="round,pad=0.3", fc=BG, ec="#DDCCCC"))

        # means
        for x_pos, grp in [(1, grp_no), (2, grp_yes)]:
            ax.scatter(x_pos, grp.mean(), marker="D",
                       color="white", s=55, zorder=5)
            ax.text(x_pos, grp.mean() + 0.25, f"μ={grp.mean():.1f}",
                    ha="center", fontsize=8, color=CHARCOAL, fontweight="bold")

        ax.set_xticklabels(
            [f"No\n(n={len(grp_no)})", f"Yes\n(n={len(grp_yes)})"],
            fontsize=9
        )
        ax.set_title(title, fontsize=10.5, color=CHARCOAL, pad=6)
        ax.set_yticks(range(1, 12))
        ax.set_yticklabels(
            [str(w) if w < 11 else "Finale" for w in range(1, 12)],
            fontsize=8
        )

    axes[0].set_ylabel("Week of Elimination", fontsize=11)

    legend_elements = [
        Patch(facecolor=TEAL, alpha=0.75, label="Feature = Yes"),
        Patch(facecolor=ROSE, alpha=0.75, label="Feature = No"),
        plt.scatter([], [], marker="D", color=CHARCOAL, s=40, label="Mean"),
    ]
    fig.legend(handles=legend_elements, fontsize=9,
               loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.04))

    plt.tight_layout()
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"  ✓ {out}")


# ════════════════════════════════════════════════════════════════════════════
# TEXT SUMMARY
# ════════════════════════════════════════════════════════════════════════════

def print_summary(df, cx, weekly):
    print("\n" + "=" * 70)
    print("  SUMMARY STATISTICS — BACHELOR / BACHELORETTE")
    print("=" * 70)

    print(f"\n  Dataset: {df.shape[0]:,} contestant-week rows")
    print(f"  Seasons: {cx.groupby(['SHOW','SEASON']).ngroups} "
          f"(Bachelor: {cx[cx['SHOW']=='Bachelor']['SEASON'].nunique()}, "
          f"Bachelorette: {cx[cx['SHOW']=='Bachelorette']['SEASON'].nunique()})")
    print(f"  Unique contestants: {len(cx):,}")
    print(f"  Reached finale:  {cx['finalist'].sum()} ({cx['finalist'].mean()*100:.1f}%)")

    print("\n  Avg elimination week (excl. finalists):")
    for show in ["Bachelor", "Bachelorette"]:
        sub = cx[(cx["SHOW"] == show) & ~cx["finalist"]]
        print(f"    {show}: {sub['elim_week'].mean():.2f}  "
              f"(SD={sub['elim_week'].std():.2f})")

    print("\n  Feature prevalence across all contestants:")
    for col, lbl in [("ever_date_rose","Date rose"),("ever_one_on_one","1-on-1"),
                     ("first_impression","1st imp. rose"),("ever_has_date","Any date")]:
        print(f"    {lbl:20s}: {cx[col].mean()*100:.1f}%")

    print("\n  Peak/trough elimination rate by week:")
    print(f"    Highest: Week {weekly.loc[weekly['elim_rate'].idxmax(),'WEEK']}  "
          f"({weekly['elim_rate'].max()*100:.1f}%)")
    print(f"    Lowest:  Week {weekly.loc[weekly['elim_rate'].idxmin(),'WEEK']}  "
          f"({weekly['elim_rate'].min()*100:.1f}%)")
    print("=" * 70 + "\n")


# ════════════════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import os
    DATA = "/mnt/user-data/uploads/contestant_weekly.csv"
    OUT  = "/mnt/user-data/outputs"
    os.makedirs(OUT, exist_ok=True)

    print("\nLoading data …")
    df, cx, weekly, alive = load(DATA)

    print("Generating Figure 1 — Summary statistics …")
    fig1_summary(df, cx, f"{OUT}/stat_fig1_summary.png")

    print("Generating Figure 2 — Survival curves …")
    fig2_survival(alive, cx, f"{OUT}/stat_fig2_survival.png")

    print("Generating Figure 3 — Weekly dynamics …")
    fig3_weekly_dynamics(weekly, f"{OUT}/stat_fig3_weekly.png")

    print("Generating Figure 4 — Feature box plots …")
    fig4_feature_boxplots(cx, f"{OUT}/stat_fig4_boxplots.png")

    print_summary(df, cx, weekly)
    print(f"All outputs saved to {OUT}")